In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import spearmanr

from adserve_sim.data.download import RAW_DIR, RAW_FILENAME
from adserve_sim.data.schema import (
    CATEGORICAL_COLUMNS,
    LABEL,
    RAW_HOUR,
    TIMESTAMP,
)

pd.set_option("display.max_columns", 30)
plt.rcParams["figure.figsize"] = (10, 4)

: 

# Avazu EDA

**adserve-sim** replays real ad-request logs and asks: if a click model ranks impressions correctly but predicts the wrong probabilities, what does that cost once bids compete for a slot? AUC can't see this on its own as it's invariant to any monotone rescaling of the predictions, but `bid = value x pCTR` means a systematic probability error becomes a systematic bidding error.

Everything in this notebook is currently a hardcoded assumption in `src/`, written from Avazu's published schema rather than the actual file. This notebook confirms or breaks each one against the real data. No modeling here, just looking.

Data: `data/processed/avazu_sample.parquet`, produced by
```
uv run python -m adserve_sim.data.download --sample-rows 200000
```
(a per-hour stratified sample of the ~40M-row raw file).

In [ ]:
SAMPLE_PATH = Path("../data/processed/avazu_sample.parquet")

if not SAMPLE_PATH.exists():
    raise FileNotFoundError(
        f"{SAMPLE_PATH} not found. Run:\n"
        "  uv run python -m adserve_sim.data.download --sample-rows 200000"
    )

df = pd.read_parquet(SAMPLE_PATH)
print(f"{len(df):,} rows, {df[TIMESTAMP].min()} to {df[TIMESTAMP].max()}")
df.head()

In [ ]:
df.info()

## 1. Base CTR

Synthetic fixtures used ~17% to keep small samples from being all-zero. Real
display CTR is typically far lower (0.1–0.5%) so the fixtures were expected
to be wildly optimistic.

They aren't, and the reason matters. [Kaggle's data description](https://www.kaggle.com/competitions/avazu-ctr-prediction/data) states clicks
and non-clicks were subsampled at *different* rates before release, so the
file's click rate is inflated relative to the traffic it came from.

This has a direct consequence for a project about calibration. A model fitted
here is calibrated to the **sampled** distribution, not to real traffic. Its
probabilities are correct for this file and systematically too high for the
world. That is a known, correctable bias (the fix is a log-odds offset of
`log(sampling_rate_ratio)`), but the ratio isn't published, so it can't be
undone.

What survives: *relative* calibration comparisons. Isotonic versus Platt versus
raw, and the revenue gap between calibrated and uncalibrated, are all valid
because both sides sit in the same distorted distribution. What doesn't survive
is any claim about absolute click probabilities.

In [ ]:
base_ctr = df[LABEL].mean()
n_clicks = int(df[LABEL].sum())
print(f"base CTR: {base_ctr:.4%}  ({n_clicks:,} clicks / {len(df):,} impressions)")

## 2. Cardinality per categorical

The step-3 encoding decision (CatBoost's native categorical handling vs. a hand-written out-of-fold encoder) is justified by `device_ip` / `device_id` running to "millions" of distinct values. Check that against the sample (note cardinality in a 200k-row sample is a lower bound on the full 40M-row file's true cardinality, not the answer itself).

In [ ]:
cardinality = df[list(CATEGORICAL_COLUMNS)].nunique().sort_values(ascending=False)

In [ ]:
ax = cardinality.plot(kind="bar")
ax.set_yscale("log")
ax.set_ylabel("distinct values (log scale)")
ax.set_title("Categorical cardinality, 200k-row sample")
plt.tight_layout()
plt.show()

display(cardinality)

`singleton_share` is the fraction of a column's *values* seen exactly once.
`rows_in_singletons` is the fraction of *impressions* sitting on such a value, which is
the more useful number because it says how much of the data a naive target
encoder would encode with that row's own label and nothing else.

This is the concrete justification for out-of-fold encoding in
`features/build.py`, and for the smoothing prior: a category seen once is not
evidence of a 0% or 100% click rate, it is evidence of nothing.

In [ ]:
singletons = pd.DataFrame(
    {
        "n_unique": df[col].nunique(),
        "singleton_share": (df[col].value_counts() == 1).mean(),
        "rows_in_singletons": (df[col].value_counts() == 1).sum() / len(df),
    }
    for col in CATEGORICAL_COLUMNS
).set_index(pd.Index(CATEGORICAL_COLUMNS))

singletons = singletons.sort_values("rows_in_singletons", ascending=False)

ax = singletons[["singleton_share", "rows_in_singletons"]].head(8).plot(kind="bar")
ax.set_ylabel("share")
ax.set_title("Where naive target encoding would copy the label")
plt.tight_layout()
plt.show()

display(singletons.head(8))

## 3 & 4. Does `hour` parse cleanly, and is there a real daily rhythm?

Both questions are about the `hour` column at full scale, not just the sample, and both are cheap to check with a single column-limited scan of the raw file, so they share one pass here.

**On parsing:** `prepare_sample` calls `parse_hour` once on the fully concatenated frame and raises on the first failure. Two readings of that: if `avazu_sample.parquet` exists at all, every row in the raw file already parsed successfully, the check below is a genuine confirmation (with a row count), not a vacuous one, but it can't surface *partial* failures the way a non-raising per-row scan can.

**On volume:** the stratified sampler in `download.py` preserves the hourly volume curve on the assumption that it's not flat and is therefore worth preserving. This confirms there's a real curve to preserve in the first place, and how much it varies across the 10 days.

In [ ]:
raw_path = Path("..") / RAW_DIR / RAW_FILENAME

total_rows = 0
bad_hour_values: set[str] = set()
hour_counts = pd.Series(dtype="int64")

reader = pd.read_csv(
    raw_path,
    usecols=[RAW_HOUR],
    dtype={RAW_HOUR: "str"},
    chunksize=1_000_000,
    compression="infer",
)
for chunk in reader:
    parsed = pd.to_datetime(chunk[RAW_HOUR], format="%y%m%d%H", errors="coerce")
    total_rows += len(chunk)
    bad_hour_values.update(chunk.loc[parsed.isna(), RAW_HOUR].unique())
    hour_counts = hour_counts.add(parsed.value_counts(), fill_value=0)

hour_counts = hour_counts.sort_index()
print(f"{total_rows:,} rows scanned")
print(
    f"{len(bad_hour_values)} distinct unparseable hour value(s)"
    + (f": {list(bad_hour_values)[:10]}" if bad_hour_values else "")
)

In [ ]:
ax = hour_counts.plot()
ax.set_ylabel("requests")
ax.set_title("Hourly request volume, full raw file")
plt.tight_layout()
plt.show()

daily = hour_counts.groupby(hour_counts.index.normalize()).sum()
cv = daily.std() / daily.mean()
print(f"day-to-day coefficient of variation: {cv:.3f}")
daily

## 5. Are `C14`-`C21` genuinely categorical?

Spearman's rho measures whether CTR moves *consistently* up or down with the
numeric value, without assuming the relationship is a straight line. Near +1 or
-1 means the ordering carries information; near 0 means the numbers are just
labels.

The bucket count is the catch. `C15`, `C16` and `C18` have so few distinct
values that they collapse to two buckets, and any two points lie on a monotone
line — so those panels look like the cleanest evidence in the grid while being
the only ones that cannot be evidence of anything. They are marked inconclusive
rather than scored.

In [ ]:
MIN_BUCKETS = 5

c_columns = [c for c in CATEGORICAL_COLUMNS if c.startswith("C") and c != "C1"]
results = []

fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for ax, col in zip(axes.flat, c_columns, strict=False):
    values = df[col].astype(int)
    buckets = pd.qcut(values, q=min(20, values.nunique()), duplicates="drop")
    stats = df.groupby(buckets, observed=True)[LABEL].agg(["mean", "count"])
    midpoints = [interval.mid for interval in stats.index]

    if len(stats) >= MIN_BUCKETS:
        rho, pvalue = spearmanr(midpoints, stats["mean"])
        verdict = f"rho={rho:+.2f}, p={pvalue:.3f}"
    else:
        rho, verdict = float("nan"), "too few buckets"

    results.append({"column": col, "n_buckets": len(stats), "spearman": rho})

    ax.plot(midpoints, stats["mean"], marker="o", markersize=4)
    ax.set_title(f"{col} ({len(stats)} buckets) {verdict}", fontsize=9)

fig.suptitle("Mean CTR by value-rank bucket, with rank correlation")
plt.tight_layout()
plt.show()

display(pd.DataFrame(results).set_index("column"))

All 21 categoricals are read as `str` (`RAW_DTYPES`), which is the safe default for anonymised hash IDs. But `C14`-`C21` have undisclosed meanings and Avazu's docs don't rule out them being ordinal (e.g. a bucketed count or a screen dimension) rather than a true unordered category. If a column's mean CTR moves *monotonically* with its numeric value, that's a signal it carries ordinal information that one-hot / hash encoding would throw away, and CatBoost's native handling wouldn't recover it either -- it would need to be passed as numeric instead.

In [ ]:
c_columns = [c for c in CATEGORICAL_COLUMNS if c.startswith(("C1", "C2"))]
c_columns = [c for c in c_columns if c not in ("C1",)]  # C1 documented as a plain category

fig, axes = plt.subplots(2, 4, figsize=(16, 6), sharey=False)
for ax, col in zip(axes.flat, c_columns, strict=False):
    values = df[col].astype(int)
    n_buckets = min(20, values.nunique())
    buckets = pd.qcut(values, q=n_buckets, duplicates="drop")
    bucket_stats = df.groupby(buckets, observed=True)[LABEL].agg(["mean", "count"])
    midpoints = [interval.mid for interval in bucket_stats.index]
    ax.plot(midpoints, bucket_stats["mean"], marker="o", markersize=4)
    ax.set_title(f"{col}  (n_unique={values.nunique()}, {len(bucket_stats)} buckets)", fontsize=9)
    fig.suptitle(
        "Mean CTR (Y) by value-rank bucket (X) pools impressions per bucket to see a real trend"
    )
plt.tight_layout()
plt.show()

## 6. Base rate stability across days

`split.py` cuts train/validation/test on day boundaries. That's the right call for avoiding leakage, but it's only free of a second problem -- distribution shift -- if the click-through rate doesn't move much day to day. If it does, some of what later looks like a calibration problem could just be this.

In [ ]:
daily_ctr = df.groupby(df[TIMESTAMP].dt.normalize())[LABEL].mean()
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
palette = plt.get_cmap("tab10")
color_by_weekday = dict(zip(weekday_order, [palette(i) for i in range(7)], strict=True))
bar_colors = [color_by_weekday[day] for day in daily_ctr.index.day_name()]

fig, ax = plt.subplots()
ax.bar(daily_ctr.index.strftime("%m-%d"), daily_ctr.values, color=bar_colors)
ax.set_ylabel("CTR")
ax.set_title("Daily CTR, sample")
legend_handles = [plt.Rectangle((0, 0), 1, 1, color=color_by_weekday[day]) for day in weekday_order]
ax.legend(legend_handles, weekday_order, bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.show()

In [ ]:
profile = hour_counts.groupby(hour_counts.index.hour).mean()

ax = profile.plot(marker="o")
ax.set_xlabel("hour of day")
ax.set_ylabel("mean requests")
ax.set_title("Average daily traffic profile")
ax.set_xticks(range(0, 24, 2))
plt.tight_layout()
plt.show()

print(f"peak/trough ratio: {profile.max() / profile.min():.1f}x")

The raw series above is noisy; this is the same data folded into an average
day. The peak-to-trough ratio is what a budget pacing controller has to spend
against, a campaign that spends at a flat hourly rate will exhaust early on
quiet hours and underspend at peak.

Note the timestamps carry no timezone, so "hour 0" is whatever clock Avazu
recorded in. The shape is usable; the absolute hours are not interpretable.

## Summary

| Question | Confirmed / broken |
|---|---|
| Base CTR ~17%? | Confirmed: 16.9%. Not representative of true production CTR; Kaggle's data description states clicks/non-clicks were subsampled at different rates before release, so this number is real but inflated relative to raw traffic. |
| `device_ip` / `device_id` cardinality in the millions? | partial, sample gives 143,551 / 33,074 (lower bound only) |
| `hour` parses cleanly on every row? | confirmed, 0 unparseable |
| Hourly volume curve is non-flat and worth preserving? | confirmed: CV ≈ 0.175, range 3.2M–5.3M/day, two Tuesday spikes breaking any clean weekly pattern |
| `C14`-`C21` genuinely unordered? | _fill in_ |
| Base rate stable across the 10 days? | _fill in_ |